# 02C SCD1 and SCD2 Demo

Load complete Product Master snapshots into one Bronze source, then publish the same governed Full Read to SCD1 and SCD2 
Silver targets.

### Before you begin

Run 00B and then 00C to stage Day 1 Product Master in Bronze. Run this notebook for Day 1, then change `DEMO_DAY` to 2 and 3 in order. Only the scenario cell changes Bronze; the governed read and both SCD writes remain identical. Do not rerun 00C after Day 1.


# 0. Workspace — Setup Notebook

Run `setup_notebook()` to load `00_env_config`, resolve the caller environment, validate targets, and display configuration readiness. For shared configuration, pass `config_workspace_id` here. Setup Pipeline reuses this configuration.

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    finalize_pipeline,
    orchestrate_read,
    orchestrate_write,
    profile_table,
    read_lakehouse_csv,
    read_lakehouse_table,
    setup_notebook,
    setup_pipeline,
    write_lakehouse_table,
)


context = setup_notebook()

# For shared config, use the workspace hosting 00_env_config:
# context = setup_notebook(
#     config_workspace_id="<configuration-workspace-id>",
# )


## Demo day

Start with `1`. After a successful persistent run, change only this value to `2` and rerun from the top, then repeat with `3`.


In [ ]:
DEMO_DAY = 1

if DEMO_DAY not in {1, 2, 3}:
    raise ValueError("DEMO_DAY must be 1, 2, or 3.")


## Prepare the Bronze snapshot

00C already stages Day 1. For Days 2 and 3, this single scenario cell replaces the Bronze snapshot; SCD1 and SCD2 behaviour remains inside `orchestrate_write()`.

This is explicit demo source preparation before pipeline selection. It changes Bronze independently of the later pipeline execution; the orchestrated Reads and Writes use the selected mode. Run it deliberately when advancing the demo day.


In [ ]:
if DEMO_DAY > 1:
    snapshot_df = read_lakehouse_csv(
        f"demo/products_day{DEMO_DAY}.csv",
        store="Bronze", header=True, inferSchema=True,
    )
    if snapshot_df.count() != 8:
        raise ValueError(f"Day {DEMO_DAY} must contain exactly 8 Product Master rows.")
    write_lakehouse_table(
        snapshot_df, table_name="product_master_updates", store="Bronze",
        schema="demo", mode="overwrite",
    )


# 1. Setup Pipeline

Use **Dry Run** to preview without writing, **Ungoverned** to run without Data Contracts, or **Governed** requires activated deployments and Contracts; it remains blocked until trusted deployment verification is integrated. Pipeline source participation and reporting are automatic.

Run the final Finalize cell after the complete pipeline. Dry Run returns a session-only summary; persistent modes record the actual outcome.


In [ ]:
PIPELINE_MODE = "Ungoverned"  # Dry Run, Ungoverned, Governed

pipeline = setup_pipeline(
    pipeline_mode=PIPELINE_MODE,
    profiling_enabled=True,  # Standard profiling, independently of mandatory Guardrails.
    profiling_interval_days=1,  # Shared physical-table completion history.
    profiling_max_rows=None,  # Complete table; positive integer selects bounded rows.
    grain_discovery_enabled=False,  # Opt in once per physical table.
    frequency_enabled=False,  # Opt in once per physical table.
    source_drift_enabled=False,  # Opt in to complete-source observation.
    source_drift_interval_days=1,  # Since confirmed shared source baseline acceptance.
    source_drift_block=True,  # False warns; baseline acceptance still waits for Finalize.
)


# 2. Read

Read the complete Product Master snapshot. SCD1 and SCD2 processing remains configured per target.


In [ ]:
# Keep complete Read results for transformation and lineage.
sources = {}


## READ 1 — Product Master


In [ ]:
source = orchestrate_read(

    store="Bronze",
    schema="demo",
    table_name="product_master_updates",
    read_mode="full",
    query=None,

    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["product_master"] = source

# Inspect the actual rows returned by this Read.
display(sources["product_master"]["dataframe"].limit(10))


# 3. Transform


**Optional — Profile your transformation:** FabricOps already profiles governed Reads and Writes according to pipeline settings. Uncomment the example below to inspect a transformed DataFrame before writing. DataFrame-only profiling does not persist governed profile metadata; grain discovery and frequency profiling are off by default.


In [ ]:
transformed_df = (
    sources["product_master"]["dataframe"]
    .select(
        "product_id",
        "product_name",
        "product_category",
        F.col("list_price").cast("double").alias("list_price"),
        F.col("modified_datetime").cast("timestamp").alias("modified_datetime"),
    )
)

# Optional development inspection.
# display(transformed_df)

# Optional: profile the transformed DataFrame before writing (no metadata persistence).
# transformed_profile = profile_table(dataframe=transformed_df)


# 4. Write

Both targets receive the same transformed DataFrame and governed Read result. Each Write is an independent publication.

### Choose current truth or historical truth

**SCD1 — Maintain current truth:** update existing business keys and insert new keys when only the latest value matters. Previous values are not retained as historical versions. Use for current-state reference and master data.

**SCD2 — Preserve historical truth:** close previous versions and insert new versions when tracked values change. `_is_current` identifies current versions. Use for audit, historical reporting and time-based analysis.

| Scenario | Recommended strategy |
|---|---|
| Only latest customer address matters | SCD1 |
| Need to know a customer's previous address | SCD2 |
| Product category mapping should always show latest classification | SCD1 |
| Need historical product classifications for past reporting | SCD2 |

Business keys identify stable entities: this demo uses unique, non-null `product_id` values. SCD2 uses `modified_datetime` as the effective date and tracks only `product_category` and `list_price`. A changed version must have an effective time later than the current version start. Unchanged tracked values do not create a version, even when the snapshot timestamp changes. Select tracked columns deliberately: changes only to untracked business fields do not create a new version or refresh those stored fields. Neither strategy deletes missing keys solely because a snapshot omits them.


In [ ]:
# Keep complete Write results for optional inspection.
writes = {}


## WRITE 1 — SCD1 current state


In [ ]:
writes["product_master_scd1"] = orchestrate_write(
    transformed_df,

    store="Silver",
    schema="demo",
    table_name="product_master_scd1",
    write_mode="scd1",
    write_parameters={"key_columns": ["product_id"]},

    repartition_by=None,

    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)

# Inspect the final prepared Write input, including FabricOps audit and processing columns.
display(writes["product_master_scd1"]["dataframe"].limit(10))

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(writes["product_master_scd1"].get("dq_result"))
# display((writes["product_master_scd1"].get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the final published target after a successful persistent execution.
# from fabricops_kit import read_lakehouse_table
# display(read_lakehouse_table(store="Silver", schema="demo", table_name="product_master_scd1").orderBy("product_id").limit(10))


## WRITE 2 — SCD2 history


In [ ]:
writes["product_master_scd2"] = orchestrate_write(
    transformed_df,

    store="Silver",
    schema="demo",
    table_name="product_master_scd2",
    write_mode="scd2",
    write_parameters={
        "key_columns": ["product_id"],
        "effective_column": "modified_datetime",
        "tracked_columns": ["product_category", "list_price"],
    },

    repartition_by=None,

    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)

# Inspect the final prepared Write input, including FabricOps audit and processing columns.
display(writes["product_master_scd2"]["dataframe"].limit(10))

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(writes["product_master_scd2"].get("dq_result"))
# display((writes["product_master_scd2"].get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the final published target after a successful persistent execution.
# from fabricops_kit import read_lakehouse_table
# display(read_lakehouse_table(store="Silver", schema="demo", table_name="product_master_scd2").orderBy("product_id", "_effective_from").limit(10))


## Expected results and reset

Persisted row counts require Ungoverned or Governed publication. Dry Run returns diagnostics without updating Silver or processing state. Ungoverned needs no Data Contract but cannot write a table that is already governed. After all stages, call `finalize_pipeline(pipeline)` before advancing the demo day.

| Day | SCD1 rows | SCD2 versions |
|---:|---:|---:|
| 1 | 8 | 8 |
| 2 | 8 | 9 |
| 3 | 8 | 10 |

For a complete lifecycle restart, stop current and earlier demo jobs and run the single 00C reset cell with `RESET_DEMO = True`. This clears all metadata and native lifecycle artifacts in the dedicated `demo_metadata` item, restores five Day 1 sources and removes six outputs. Wait for `RESET COMPLETE` before Step 1, then set both 02B and 02C to `DEMO_DAY = 1`. Never reset between days or delete checkpoints separately.

For same-session retries, source-staging diagnosis and safe checkpoint reconciliation, see the [Guided Demo troubleshooting procedure](https://voycepeh.github.io/FabricOps-Starter-Kit/guided-demo/02C-build-and-run-scd-etl/#diagnose-a-later-day-before-changing-the-scd-targets). Keep existing checkpoints when repairing a mismatched target; never use a demo reset as production recovery.


# 5. Finalize

Complete the run after every Read, transformation and Write. Dry Run keeps its summary in the session only. Ungoverned and Governed record the real outcome; Governed also releases its activation reservation. A stopped notebook remains incomplete.

In [ ]:
# 5. Finalize
pipeline_result = finalize_pipeline(pipeline)